In [1]:
import phoenix as px
from phoenix.otel import register
from openinference.instrumentation.langchain import LangChainInstrumentor

# 1. Запускаем сам сервер Phoenix
session = px.launch_app()

# 2. Регистрируем провайдер трассировки (OpenTelemetry)
# Он будет перехватывать данные и отправлять их в локальный Phoenix
tracer_provider = register()

# 3. Включаем "прослушку" именно для LangChain
LangChainInstrumentor().instrument(tracer_provider=tracer_provider)

print(f"Phoenix готов! Дашборд тут: {session.url}")

/Users/artemzmailov/Desktop/kitoboy-PII/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/Users/artemzmailov/.local/share/uv/python/cpython-3.11.15-macos-aarch64-none/lib/python3.11/contextlib.py:144: SAWarning: Skipped unsupported reflection of expression-based index ix_cumulative_llm_token_count_total
  next(self.gen)
/Users/artemzmailov/.local/share/uv/python/cpython-3.11.15-macos-aarch64-none/lib/python3.11/contextlib.py:144: SAWarning: Skipped unsupported reflection of expression-based index ix_latency
  next(self.gen)


🌍 To view the Phoenix app in your browser, visit http://localhost:6006/
📖 For more information on how to use Phoenix, check out https://arize.com/docs/phoenix
🔭 OpenTelemetry Tracing Details 🔭
|  Phoenix Project: default
|  Span Processor: SimpleSpanProcessor
|  Collector Endpoint: localhost:4317
|  Transport: gRPC
|  Transport Headers: {}
|  
|  Using a default SpanProcessor. `add_span_processor` will overwrite this default.
|  
|  ⚠️ WARNING: It is strongly advised to use a BatchSpanProcessor in production environments.
|  
|  `register` has set this TracerProvider as the global OpenTelemetry default.
|  To disable this behavior, call `register` with `set_global_tracer_provider=False`.

Phoenix готов! Дашборд тут: http://localhost:6006/


In [2]:
import random
from pathlib import Path
from faker import Faker

ROOT = Path.cwd()
ENTITIES_POOL_DIR = ROOT / ".." / "outputs"
HISTORY_DIR = ROOT / ".." / "outputs" / "history"


SEED = 42
N_NAMES = 10000
N_ADDRESSES = 10000

random.seed(SEED)
fake = Faker("ru_RU")
Faker.seed(SEED)

print(ROOT)


/Users/artemzmailov/Desktop/kitoboy-PII


In [3]:
def make_full_name(gender: str) -> dict[str, str]:
    if gender == "male":
        last = fake.last_name_male()
        first = fake.first_name_male()
        middle = fake.middle_name_male()
    else:
        last = fake.last_name_female()
        first = fake.first_name_female()
        middle = fake.middle_name_female()

    return {
        'first': first,
        'middle': middle,
        'last': last
    }


full_names_pool = []
while len(full_names_pool) < N_NAMES:
    gender = random.choice(["male", "female"])
    name = make_full_name(gender)
    full_names_pool.append(name)

#print(full_names_pool)


In [4]:
def name_surface_variants(item: dict[str, str]) -> list[str]:
    last = item["last"]
    first = item["first"]
    middle = item["middle"]

    variants = [
        f"{last} {first} {middle}",
        f"{first} {middle} {last}",
        f"{first} {last}",
        f"{last} {first}",
        f"{last} {first[0]}.{middle[0]}.",
        f"{first} {middle}",
        f"{last.upper()} {first} {middle}",
        f"{last.lower()} {first.lower()} {middle.lower()}",
    ]
    return variants

names_pool = set()
for item in full_names_pool:
    variants = name_surface_variants(item)
    names_pool.update(variants)

# print(names_pool)


In [5]:
def make_building() -> str:
    building = str(random.randint(1, 180))
    suffix = random.choices(
        ['', f'/{random.randint(1, 9)}', f' к {random.randint(1, 6)}', f' стр {random.randint(1, 5)}'],
        weights=[0.72, 0.16, 0.08, 0.04],
        k=1,
    )[0]
    return building + suffix


def make_address() -> dict[str, str | int | None]:
    has_flat = random.random() < 0.82
    return {
        "region": fake.region().strip(),
        "city": fake.city_name().strip(),
        "street": fake.street_name().strip(),
        "building": make_building(),
        "flat": random.randint(1, 250) if has_flat else None,
        "postcode": fake.postcode().strip(),
    }


def address_to_canonical(item: dict[str, str | int | None]) -> str:
    flat = f", кв. {item['flat']}" if item.get("flat") is not None else ""
    return f"{item['region']}, г. {item['city']}, {item['street']}, д. {item['building']}{flat}, {item['postcode']}"


full_addresses_pool = []
while len(full_addresses_pool) < N_ADDRESSES:
    item = make_address()
    full_addresses_pool.append(item)

# print(full_addresses_pool)


In [6]:
def address_surface_variants(item: dict[str, str | int | None]) -> list[str]:
    region = str(item["region"])
    city = str(item["city"])
    street = str(item["street"])
    building = str(item["building"])
    flat = item.get("flat")
    postcode = str(item["postcode"])

    flat_part = f", кв. {flat}" if flat is not None else ""
    flat_short = f" кв {flat}" if flat is not None else ""
    flat_dash = f"-{flat}" if flat is not None else ""
    flat_slash = f"/{flat}" if flat is not None else ""

    variants = [
        f"г. {city}, {street}, д. {building}{flat_part}",
        f"{city}, {street}, {building}{flat_dash}",
        f"{city} {street} {building}{flat_slash}",
        f"{postcode}, {city}, {street}, {building}{flat_short}",
        f"{street}, д {building}{flat_short}, {city}",
        f"{city.lower()}, {street.lower()} {building}{flat_dash}",
        f"{street} {building}{flat_short}",
    ]

    # Регион полезен как отдельный сигнал, но Faker не согласует его с городом.
    # Поэтому добавляем его редко, чтобы не засорять пул географически невозможными адресами.
    if random.random() < 0.05:
        variants.append(f"{region}, {city}, {street}, {building}{flat_short}")

    return variants


address_pool = set()

for item in full_addresses_pool:
    variants = address_surface_variants(item)
    address_pool.update(variants)

# print(address_pool)


In [11]:
import os
import json
from datetime import datetime
from typing import List

from dotenv import load_dotenv
from pydantic import BaseModel, Field
from langchain_mistralai import ChatMistralAI
from langchain_core.messages import SystemMessage, HumanMessage

load_dotenv()

class Entities(BaseModel):
    items: List[str] = Field(description="Список зашумленных сущностей")

llm = ChatMistralAI(
    model="mistral-small-latest",
    temperature=0.7,
    timeout=60,
    max_retries=3,
)
structured_llm = llm.with_structured_output(Entities)


In [12]:
PROMPTS = {
    "FULL_NAME": """
Роль: Ты помогаешь готовить синтетический PII-датасет для проверки NER-системы.

Твоя задача: зашумить входные ФИО так, как их мог бы написать реальный пользователь в сообщении.
Для каждого входного значения верни один новый вариант записи.

Что можно делать:
1. Менять порядок частей ФИО:
   - Иванов Иван Иванович -> Иван Иванович Иванов
   - Иванов Иван Иванович -> Иван Иванов
   - Иванов Иван Иванович -> Иванов Иван
2. Использовать инициалы:
   - Иванов Иван Иванович -> Иванов И. И.
   - Иванов Иван Иванович -> И. И. Иванов
   - Иванов Иван Иванович -> Иванов И.И.
3. Менять регистр:
   - иванов иван иванович
   - ИВАНОВ Иван Иванович
4. Добавлять/убирать точки и лишние пробелы:
   - Иванов И И
   - Иванов  И.И.
5. Делать естественные короткие формы, если они сохраняют того же человека:
   - Мария Александровна Петрова -> Мария Петрова
   - Петрова Мария Александровна -> Петрова М. А.

Строгие ограничения:
1. Не меняй имя, фамилию и отчество на другие.
2. Не придумывай новые ФИО.
3. Не добавляй должности, организации, адреса, телефоны, email или комментарии.
4. Не превращай значение в предложение. Нужна только сама сущность.
5. Не возвращай только голые инициалы без фамилии или имени, например "И. П.".
6. Верни примерно столько же items, сколько значений пришло на вход.
7. Верни только JSON по схеме Entities: {"items": [...]}.
""".strip(),

    "ADDRESS": """
Роль: Ты помогаешь готовить синтетический PII-датасет для проверки NER-системы.

Твоя задача: зашумить входные адреса так, как их мог бы написать реальный пользователь в сообщении.
Для каждого входного значения верни один новый вариант записи.

Что можно делать:
1. Менять порядок частей адреса:
   - Москва, ул. Тверская, д. 8, кв. 12 -> Тверская 8 кв 12, Москва
   - г. Казань, Баумана, д. 15 -> Казань Баумана 15
2. Использовать бытовые сокращения:
   - город -> г
   - улица -> ул
   - дом -> д
   - квартира -> кв
   - корпус -> к / корп
3. Убирать часть служебной пунктуации:
   - г. Москва, ул. Ленина, д. 10, кв. 5 -> москва ленина 10 кв 5
4. Менять регистр, пробелы и разделители:
   - СПб, Невский 12-34
   - спб невский д12 кв34
   - Екатеринбург/Мира/7/22
5. Делать более разговорные, но короткие формы:
   - Москва, Тверская 8-12
   - Питер, Невский 15 кв 9
   - Казань Баумана 11/45

Строгие ограничения:
1. Не меняй смысл адреса.
2. Не меняй цифры дома, квартиры, корпуса и индекса.
3. Не меняй город и улицу на другие.
4. Регион можно сохранить, переставить или опустить, если без него адрес выглядит естественнее. Не придумывай новый регион.
5. Не добавляй ФИО, телефоны, email или любые другие PII-сущности.
6. Не превращай значение в длинное предложение. Нужна только сама address-сущность.
7. Если во входе есть индекс, желательно сохранить его, но можно переставить в другое место.
8. Верни примерно столько же items, сколько значений пришло на вход.
9. Верни только JSON по схеме Entities: {"items": [...]}.
""".strip(),
}


def noise_batch(entity_name: str, batch: list[str]) -> list[str]:
    messages = [
        SystemMessage(content=PROMPTS[entity_name]),
        HumanMessage(content=json.dumps(batch, ensure_ascii=False, indent=2)),
    ]
    return structured_llm.invoke(messages).items


In [13]:
def collect_noised_pool(entity_name: str, source_pool, count: int, batch_size: int = 10, save_every: int = 100):
    path = '../outputs/'
    source = list(source_pool)
    samples = []
    total_cnt = 0

    while total_cnt < count:
        current_batch_size = min(batch_size, count - total_cnt)
        batch = random.sample(source, k=min(current_batch_size, len(source)))
        generated = noise_batch(entity_name, batch)
        samples.extend(generated)
        total_cnt += len(generated)

        if len(samples) >= save_every or total_cnt >= count:
            now = datetime.now().replace(microsecond=0)
            now_str = str(now).replace(' ', '_')
            print(now)
            print('../outputs/history/' + f'{entity_name}_NOISED_{now_str}.json')

            with open('../outputs/history/' + f'{entity_name}_NOISED_{now_str}.json', 'w') as f:
                json.dump(samples, f, ensure_ascii=False, indent=4)

            for i, val in enumerate(samples):
                print(f'index: {i}, value : {val}')

            raw = input('Delete indices: ').strip()
            delete_idx = set(map(int, raw.split())) if raw else set()
            samples_cleaned = [sample.strip() for i, sample in enumerate(samples) if i not in delete_idx]

            pool_path = path + f'{entity_name}.json'
            if os.path.exists(pool_path):
                with open(pool_path, 'r') as f:
                    old = json.load(f)
            else:
                old = []

            print('Current len:', len(old))
            new = set(old)
            new.update(samples_cleaned)
            print('Total unique count:', len(new))

            with open(pool_path, 'w') as f:
                json.dump(list(new), f, ensure_ascii=False, indent=4)

            samples = []


In [ ]:
# Примеры запуска:
# collect_noised_pool('FULL_NAME', names_pool, count=1000)
collect_noised_pool('ADDRESS', address_pool, count=4000)


2026-04-29 16:41:21
entities_pool/history/FULL_NAME_NOISED_2026-04-29_16:41:21.json
index: 0, value : Данила Егорович Красильников
index: 1, value : ЧЕРНОВА А. Г.
index: 2, value : Белякова С. А.
index: 3, value : Анна Болеславовна Лапина
index: 4, value : Майя Ильинична Волкова
index: 5, value : Синклитикия Максимовна Емельянова
index: 6, value : Александра Георгиевна Кудрявцева
index: 7, value : Дарья Романовна Казакова
index: 8, value : Ирина Антонова
index: 9, value : Наина Никитина
index: 10, value : Фомин Аникей Федотович
index: 11, value : Иннокентий Антипович
index: 12, value : Владилен Артемьевич Филатов
index: 13, value : Виктор Зыков
index: 14, value : Панфилов Силантий Теймуразович
index: 15, value : Данилов Валерьян Данилович
index: 16, value : Ян Феоктистович Игнатов
index: 17, value : Ираида Аскольдовна
index: 18, value : Игнатова Василиса Кирилловна
index: 19, value : Власович Эммануил
index: 20, value : Хохлова М.
index: 21, value : Мартынов Х.
index: 22, value : Герас

Delete indices:  19 42 45 48 52 62 67 78


Current len: 280
Total unique count: 372
2026-04-29 16:42:26
entities_pool/history/FULL_NAME_NOISED_2026-04-29_16:42:26.json
index: 0, value : Маргарита Львовна Ширяева
index: 1, value : Аверьян Ларионов
index: 2, value : Куприян Марсович Федосеев
index: 3, value : Капустин В.Б.
index: 4, value : Любосмысл Лыткин
index: 5, value : Глафира Михайловна
index: 6, value : Зоя Николаевна Белова
index: 7, value : Моисей Авдеевич Устинов
index: 8, value : Давыд Потапов
index: 9, value : Любомир Моисеев
index: 10, value : ТЕРЕНТЬЕВА Лукия Тарасовна
index: 11, value : Жданова Л. Ф.
index: 12, value : Смирнов П.
index: 13, value : Никонова Юлия М.
index: 14, value : ЗИМИНА А. М.
index: 15, value : Кириллов Р.
index: 16, value : Лыткина Надежда Леонидовна
index: 17, value : Василиса Львовна Медведева
index: 18, value : Орлов М. Ф.
index: 19, value : Пономарёв Дорофей З.
index: 20, value : Яков Ф. Колесников
index: 21, value : Александра Константинова
index: 22, value : Пахом В. Носков
index: 23, v

Delete indices:  31 43 55 71 77 90 92


Current len: 372
Total unique count: 465
2026-04-29 16:44:43
entities_pool/history/FULL_NAME_NOISED_2026-04-29_16:44:43.json
index: 0, value : Пономарева Лора
index: 1, value : Лукия Г. Голубева
index: 2, value : Цветкова Е. М.
index: 3, value : Олимпиада Евсеева
index: 4, value : Вишняков Н. А.
index: 5, value : Антонина Дроздова
index: 6, value : Фомина А. И.
index: 7, value : Субботин С. Д.
index: 8, value : Потапова И. Д.
index: 9, value : Макарова Елизавета Александровна
index: 10, value : Анатолий Глебович Сафонов
index: 11, value : Милий Герасимович Ильин
index: 12, value : Пелагея Воронцова
index: 13, value : Мстислав Всеволодович Морозов
index: 14, value : Валерьян Витальевич Гурьев
index: 15, value : Селиверст Петухов
index: 16, value : Василиса Шубина
index: 17, value : Зуев Сильвестр Адрианович
index: 18, value : Татьяна Владиславовна Кононова
index: 19, value : Радим Феофанович Аксенов
index: 20, value : Ольга Р. Стрелкова
index: 21, value : Творимир Цветков
index: 22, val